# MUSE Benchmark for spatial mutliomcis data integration on simulated dataset

Notebook benchmarks spatial mutliomcis data integration using MUSE on simulated dataset.

## Loading

In [ ]:
import muse_sc as muse
import phenograph
from sklearn.decomposition import PCA
import numpy as np
import tensorflow as tf
tf.get_logger().setLevel('ERROR')
np.random.seed(0)
import matplotlib.pyplot as plt
import os
import scanpy as sc

In [ ]:
latent_dim = 100
num_cluster = 10
sample_size = 1000
latent_code_dim = 30
observed_data_dim = 500
sigma_1 = 0.1  
sigma_2 = 0.1
decay_coef_1 = 0.5 
decay_coef_2 = 0.1
merge_prob = 0.7

In [ ]:
from matplotlib import rcParams
rcParams["figure.dpi"] = 300
rcParams["savefig.dpi"] = 300
rcParams["savefig.transparent"] = True
rcParams["figure.facecolor"] = 'white'
rcParams["axes.facecolor"] = 'white'

## MUSE pipeline

In [ ]:
# Set the directory for the datasets and the output directory
# os.chdir('/large_storage/zhoulab/shengmao/STARNet/')
data_dir = 'Original_Simulated_Data'
output_dir = 'Processed_Simulated_Data'
os.makedirs(output_dir, exist_ok=True)

# Loop through each dataset
for i in range(1, 6):
    ##check the presence of output file
    if os.path.exists(f'{output_dir}/Simulated_Dataset_{i}/MUSE_multiomics.h5ad'):
        print(f"Output for Simulated_Dataset_{i} already exists. Skipping processing.")
        continue
    print(f"Process {data_dir}/Simulated_Dataset_{i}/SimulatedData_{i} data.")
    # Read the RNA and ATAC datasets
    adata_omics1 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata_omics2 = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_atac.h5ad')
    adata_omics2.obsm['spatial'] = adata_omics1.obsm['spatial']
    adata_omics1.var_names_make_unique()
    adata_omics2.var_names_make_unique()

    # Preprocess RNA data
    sc.pp.filter_genes(adata_omics1, min_cells=10)
    # Preprocess ATAC data
    sc.pp.filter_genes(adata_omics2, min_cells=10)
    data_a = adata_omics1.X.A
    data_b = adata_omics2.X.A
    view_a_feature = PCA(n_components=latent_dim).fit_transform(data_a)
    view_b_feature = PCA(n_components=latent_dim).fit_transform(data_b)
    view_a_label, _, _ = phenograph.cluster(view_a_feature)
    view_b_label, _, _ = phenograph.cluster(view_b_feature)

    muse_feature, reconstruct_x, reconstruct_y, \
    latent_x, latent_y = muse.muse_fit_predict(data_a,
                                           data_b,
                                           view_a_label,
                                           view_b_label,
                                           latent_dim=100,
                                           n_epochs=500,
                                           lambda_regul=5,
                                           lambda_super=5)
    # Copy the results to the RNA dataset
    adata = sc.read_h5ad(f'{data_dir}/Simulated_Dataset_{i}/SimulatedData_{i}_rna.h5ad')
    adata.obsm['MUSE'] = muse_feature


    # Perform clustering
    sc.pp.neighbors(adata, n_neighbors=15, n_pcs=adata.obsm['MUSE'].shape[1],
                    use_rep='MUSE')
    sc.tl.leiden(adata, resolution=0.6)

    # Plot the spatial clustering results
    sc.pl.spatial(adata, color=['cell_type', 'leiden'], spot_size=0.12, wspace=0.2)

    # Save the processed dataset
    adata.write_h5ad(f'{output_dir}/Simulated_Dataset_{i}/MUSE_multiomics.h5ad', compression='gzip')

In [ ]:
!pip list